In [ ]:
!pip install sentencepiece

In [ ]:
import numpy as np
import pandas as pd

import sentencepiece as spm
import os
import json
import itertools

In [ ]:
def map_csv_to_text(csv_path_parsed, main_csv):
    df = pd.read_csv(csv_path_parsed)
    df_main = pd.read_csv(main_csv)

    drop_indices = []

    for row in df_main.itertuples(index=True):
        if row.changes in [
            "{",
        ]:
            drop_indices.append(row.Index)

    df_main.drop(index=drop_indices, inplace=True)

    df["parsed_effects"].to_csv("parsed_effect.txt", header=False, index=False)

    df_main["changes"].to_csv("changes.txt", header=False, index=False)

In [ ]:
map_csv_to_text('/kaggle/input/parsed-dna-mutations/parsed_full.csv', '/kaggle/input/all-uniprot-isodata/merged.csv')

In [ ]:
spm.SentencePieceTrainer.Train(
    '--input=/kaggle/working/parsed_effect.txt '
    '--model_prefix=input_effect_text '
    '--vocab_size=900 '
    '--character_coverage=1.0 '
    '--model_type=bpe '
    '--pad_id=0 '
    '--unk_id=3'
)

In [ ]:
spm.SentencePieceTrainer.Train(
    '--input=/kaggle/working/changes.txt '
    '--model_prefix=changes '
    '--vocab_size=900 '
    '--character_coverage=1.0 '
    '--model_type=bpe '
    '--pad_id=0 '
    '--unk_id=3'
)

In [ ]:
alphabet = ['A', 'C', 'G', 'T']
k = 6

# Special tokens with fixed IDs
special_tokens = {
    '<pad>': 0,
    '<unk>': 3,
    '<s>': 1,
    '</s>': 2
}

all_kmers = [''.join(p) for p in itertools.product(alphabet, repeat=k)]

# Map kmers to IDs starting after special tokens
start_idx = len(special_tokens)
kmer_to_id = {kmer: idx + start_idx for idx, kmer in enumerate(all_kmers)}

token_to_id = {**special_tokens, **kmer_to_id}

In [ ]:
os.makedirs('/kaggle/working/tokenizers', exist_ok=True)

In [ ]:
with open("/kaggle/working/tokenizers/vocab_protein.json", "w") as file:
    json.dump(token_to_id, file, indent=4)

In [ ]:
os.rename('/kaggle/working/changes.model', '/kaggle/working/tokenizers/changes.model')
os.rename('/kaggle/working/changes.vocab', '/kaggle/working/tokenizers/changes.vocab')
os.rename('/kaggle/working/input_effect_text.model', '/kaggle/working/tokenizers/input_effect_text.model')
os.rename('/kaggle/working/input_effect_text.vocab', '/kaggle/working/tokenizers/input_effect_text.vocab')

os.remove('/kaggle/working/parsed_effect.txt')
os.remove('/kaggle/working/changes.txt')